# Notebook 4: RAG (Retrieval-Augmented Generation) con AWS Bedrock

En este notebook aprenderás a:
- Entender el patrón RAG y por qué mejora las respuestas de los LLMs
- Construir un pipeline RAG completo desde cero
- Ingestar documentos, generar embeddings y almacenarlos en FAISS
- Recuperar contexto relevante e inyectarlo en el prompt
- Evaluar la diferencia entre respuestas con y sin RAG

---

> **Requisito:** Haber completado el Notebook 3 o tener boto3 y faiss-cpu instalados.

## 1. ¿Qué es RAG y por qué usarlo?

Los LLMs tienen dos limitaciones clave:
1. **Conocimiento estático**: solo saben lo que vieron durante el entrenamiento
2. **Alucinaciones**: pueden inventar hechos con confianza

**RAG (Retrieval-Augmented Generation)** resuelve esto en tres pasos:

```
Pregunta del usuario
       │
       ▼
  [1. RETRIEVE]  →  Buscar documentos relevantes en la base vectorial
       │
       ▼
  [2. AUGMENT]   →  Añadir esos documentos al prompt como contexto
       │
       ▼
  [3. GENERATE]  →  El LLM responde basándose en el contexto recuperado
```

## 2. Instalación de dependencias

In [ ]:
# !pip install boto3 faiss-cpu numpy --quiet

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
AWS_BEARER_TOKEN_BEDROCK=os.getenv("AWS_BEARER_TOKEN_BEDROCK")
AWS_ACCESS_KEY_ID=os.getenv("AWS_ACCESS_KEY_ID")
AWS_SECRET_ACCESS_KEY=os.getenv("AWS_SECRET_ACCESS_KEY")
AWS_DEFAULT_REGION=os.getenv("AWS_DEFAULT_REGION")

# API key de Gemini
# API_KEY = userdata.get('GEMINI_API_KEY')

## 3. Setup del cliente

In [ ]:
import boto3
import json
import numpy as np
import faiss
import textwrap

REGION           = AWS_DEFAULT_REGION
EMBEDDING_MODEL   = "amazon.titan-embed-text-v2:0"
GENERATION_MODEL  = "eu.anthropic.claude-haiku-4-5-20251001-v1:0"

bedrock_runtime = boto3.client(
    service_name="bedrock-runtime",
    region_name=REGION
)

print("✅ Cliente Bedrock listo")

✅ Cliente Bedrock listo


## 4. Funciones base: embeddings y generación

In [ ]:
def generar_embedding(texto: str) -> np.ndarray:
    """Genera un embedding con Amazon Titan Embeddings."""
    body = {"inputText": texto}
    response = bedrock_runtime.invoke_model(
        modelId=EMBEDDING_MODEL,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )
    resultado = json.loads(response["body"].read())
    return np.array(resultado["embedding"], dtype=np.float32)


def invocar_claude(prompt: str, system: str = None, max_tokens: int = 1024) -> str:
    """Invoca Claude en Bedrock y devuelve el texto generado."""
    body = {
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "temperature": 0.0,  # Determinista para RAG
        "messages": [{"role": "user", "content": prompt}]
    }
    if system:
        body["system"] = system

    response = bedrock_runtime.invoke_model(
        modelId=GENERATION_MODEL,
        body=json.dumps(body),
        contentType="application/json",
        accept="application/json"
    )
    resultado = json.loads(response["body"].read())
    return resultado["content"][0]["text"]

print("✅ Funciones base definidas")

✅ Funciones base definidas


## 5. PASO 1 — Ingesta: preparar los documentos

En un sistema real los documentos vendrían de S3, PDFs, bases de datos, etc. Aquí usamos un conjunto de documentos de política interna de una empresa ficticia.

In [ ]:
# Documentos de política interna de empresa ficticia "TechCorp"
documentos_raw = [
    {
        "id": "POL-001",
        "titulo": "Política de Vacaciones",
        "contenido": "Los empleados de TechCorp tienen derecho a 22 días hábiles de vacaciones anuales. "
                     "Las vacaciones deben solicitarse con al menos 2 semanas de antelación a través del portal de RRHH. "
                     "Los empleados con más de 5 años en la empresa tienen derecho a 27 días. "
                     "No es posible acumular más de 10 días no disfrutados al siguiente año."
    },
    {
        "id": "POL-002",
        "titulo": "Política de Teletrabajo",
        "contenido": "TechCorp permite hasta 3 días de teletrabajo a la semana para todos los empleados con más de 6 meses de antigüedad. "
                     "El teletrabajo debe acordarse con el manager directo. "
                     "Los empleados en periodo de prueba (primeros 3 meses) deben trabajar presencialmente 5 días a la semana. "
                     "Se requiere conexión a VPN corporativa en todo momento durante el teletrabajo."
    },
    {
        "id": "POL-003",
        "titulo": "Beneficios y Compensación",
        "contenido": "TechCorp ofrece seguro médico privado para el empleado y su familia desde el primer día. "
                     "Además, hay un bonus anual de hasta el 15 porciento del salario bruto basado en objetivos individuales y de empresa. "
                     "Los empleados reciben 500€ anuales para formación y certificaciones profesionales. "
                     "También se ofrece ticket restaurante de 11€ por día trabajado presencialmente."
    },
    {
        "id": "POL-004",
        "titulo": "Política de Seguridad Informática",
        "contenido": "Todos los empleados deben usar contraseñas de al menos 12 caracteres con mayúsculas, números y símbolos. "
                     "Las contraseñas deben cambiarse cada 90 días. "
                     "Está prohibido instalar software no autorizado en equipos corporativos. "
                     "Cualquier incidente de seguridad debe reportarse al equipo de IT en menos de 1 hora. "
                     "El acceso a sistemas críticos requiere autenticación de doble factor (2FA)."
    },
    {
        "id": "POL-005",
        "titulo": "Proceso de Evaluación de Desempeño",
        "contenido": "TechCorp realiza evaluaciones de desempeño dos veces al año: en junio y diciembre. "
                     "Cada empleado debe fijar entre 3 y 5 objetivos SMART al inicio de cada semestre con su manager. "
                     "La evaluación incluye autoevaluación, evaluación del manager y feedback de pares (360°). "
                     "Los resultados de la evaluación impactan directamente en el bonus anual y en las decisiones de promoción."
    },
    {
        "id": "POL-006",
        "titulo": "Gastos y Viajes de Empresa",
        "contenido": "Los viajes de empresa deben aprobarse por el manager con al menos 1 semana de antelación. "
                     "El límite de hotel es de 150€ por noche en ciudades nacionales y 200€ en internacionales. "
                     "Los vuelos deben reservarse en clase turista para trayectos menores de 5 horas. "
                     "Los gastos deben justificarse con factura y enviarse al departamento de finanzas en los 15 días posteriores al viaje."
    },
    {
        "id": "POL-007",
        "titulo": "Onboarding de Nuevos Empleados",
        "contenido": "El proceso de onboarding en TechCorp dura 30 días. "
                     "Durante la primera semana el nuevo empleado recibe formación sobre cultura, herramientas y procesos. "
                     "Se asigna un 'buddy' (compañero mentor) durante los primeros 3 meses. "
                     "Al final del primer mes se realiza una reunión de seguimiento con RRHH y el manager directo."
    },
]

print(f"📄 Total de documentos cargados: {len(documentos_raw)}")
for doc in documentos_raw:
    print(f"  [{doc['id']}] {doc['titulo']}")

📄 Total de documentos cargados: 7
  [POL-001] Política de Vacaciones
  [POL-002] Política de Teletrabajo
  [POL-003] Beneficios y Compensación
  [POL-004] Política de Seguridad Informática
  [POL-005] Proceso de Evaluación de Desempeño
  [POL-006] Gastos y Viajes de Empresa
  [POL-007] Onboarding de Nuevos Empleados


## 6. PASO 1 (cont.) — Chunking de documentos

En documentos largos es necesario dividirlos en fragmentos (chunks) para que quepan en el contexto del modelo y para recuperar solo la parte relevante.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


# Crear splitter UNA sola vez (mejor rendimiento)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=60,
    separators=["\n\n", "\n", ". ", "; ", " ", ""]
)


def chunk_texto(texto: str) -> list[str]:
    """
    Divide un texto en chunks usando LangChain TextSplitter.
    """
    return splitter.split_text(texto)


# Preparar todos los chunks con metadatos
chunks = []

for doc in documentos_raw:
    partes = chunk_texto(doc["contenido"])

    for i, parte in enumerate(partes):
        chunks.append({
            "id": f"{doc['id']}-chunk-{i}",
            "doc_id": doc["id"],
            "titulo": doc["titulo"],
            "texto": parte
        })

print(f"📦 Total de chunks generados: {len(chunks)}")

print("\nEjemplo de chunk:")
print(chunks[0])

📦 Total de chunks generados: 7

Ejemplo de chunk:
{'id': 'POL-001-chunk-0', 'doc_id': 'POL-001', 'titulo': 'Política de Vacaciones', 'texto': 'Los empleados de TechCorp tienen derecho a 22 días hábiles de vacaciones anuales. Las vacaciones deben solicitarse con al menos 2 semanas de antelación a través del portal de RRHH. Los empleados con más de 5 años en la empresa tienen derecho a 27 días. No es posible acumular más de 10 días no disfrutados al siguiente año.'}


## 7. PASO 1 (cont.) — Indexación en FAISS

In [ ]:
print("Generando embeddings e indexando chunks...")
embeddings_chunks = []

for i, chunk in enumerate(chunks):
    # Concatenar título + texto para dar más contexto al embedding
    texto_completo = f"{chunk['titulo']}: {chunk['texto']}"
    emb = generar_embedding(texto_completo)
    embeddings_chunks.append(emb)
    print(f"  [{i+1:2}/{len(chunks)}] ✓ {chunk['id']}")

# Crear índice FAISS
matriz = np.vstack(embeddings_chunks).astype(np.float32)
faiss.normalize_L2(matriz)

dimension = matriz.shape[1]
indice = faiss.IndexFlatIP(dimension)
indice.add(matriz)

print(f"\n✅ Índice construido: {indice.ntotal} vectores de {dimension} dimensiones")

Generando embeddings e indexando chunks...
  [ 1/7] ✓ POL-001-chunk-0
  [ 2/7] ✓ POL-002-chunk-0
  [ 3/7] ✓ POL-003-chunk-0
  [ 4/7] ✓ POL-004-chunk-0
  [ 5/7] ✓ POL-005-chunk-0
  [ 6/7] ✓ POL-006-chunk-0
  [ 7/7] ✓ POL-007-chunk-0

✅ Índice construido: 7 vectores de 1024 dimensiones


## 8. PASO 2 — Retrieval: recuperar contexto relevante

In [ ]:
def recuperar_contexto(query: str, top_k: int = 3) -> list[dict]:
    """Recupera los chunks más relevantes para una consulta."""
    emb_query = generar_embedding(query).reshape(1, -1).astype(np.float32)
    faiss.normalize_L2(emb_query)

    scores, indices = indice.search(emb_query, top_k)

    resultados = []
    for score, idx in zip(scores[0], indices[0]):
        chunk = chunks[idx].copy()
        chunk["score"] = float(score)
        resultados.append(chunk)
    return resultados


# Probar la recuperación
query_test = "¿Cuántos días de vacaciones tengo si llevo 6 años en la empresa?"
contextos = recuperar_contexto(query_test, top_k=3)

print(f"🔍 Query: '{query_test}'")
print("\nChunks recuperados:")
print("-" * 70)
for c in contextos:
    print(f"[{c['score']:.4f}] [{c['doc_id']}] {c['titulo']}")
    print(f"  {c['texto'][:120]}...")
    print()

🔍 Query: '¿Cuántos días de vacaciones tengo si llevo 6 años en la empresa?'

Chunks recuperados:
----------------------------------------------------------------------
[0.5167] [POL-001] Política de Vacaciones
  Los empleados de TechCorp tienen derecho a 22 días hábiles de vacaciones anuales. Las vacaciones deben solicitarse con a...

[0.2425] [POL-006] Gastos y Viajes de Empresa
  Los viajes de empresa deben aprobarse por el manager con al menos 1 semana de antelación. El límite de hotel es de 150€ ...

[0.1718] [POL-002] Política de Teletrabajo
  TechCorp permite hasta 3 días de teletrabajo a la semana para todos los empleados con más de 6 meses de antigüedad. El t...



## 9. PASO 3 — Augment + Generate: construir el prompt RAG

In [ ]:
SYSTEM_RAG = """
Eres un asistente de RRHH de TechCorp. Responde las preguntas de los empleados
basándote ÚNICAMENTE en los documentos de política interna proporcionados como contexto.
Si la información no está en el contexto, di claramente que no tienes esa información.
Sé preciso, claro y cita el documento fuente cuando sea relevante.
""".strip()


def construir_prompt_rag(query: str, contextos: list[dict]) -> str:
    """Construye el prompt RAG con la consulta y los documentos recuperados."""
    bloques_contexto = []
    for i, ctx in enumerate(contextos, 1):
        bloque = f"""[Documento {i} — {ctx['titulo']} ({ctx['doc_id']})]
{ctx['texto']}"""
        bloques_contexto.append(bloque)

    contexto_str = "\n\n".join(bloques_contexto)

    prompt = f"""A continuación tienes documentos de política interna de TechCorp:

--- CONTEXTO ---
{contexto_str}
--- FIN DEL CONTEXTO ---

Pregunta del empleado: {query}

Respuesta:"""
    return prompt


# Pipeline RAG completo
def rag(query: str, top_k: int = 3, verbose: bool = False) -> str:
    """Pipeline RAG completo: retrieve → augment → generate."""
    # 1. Retrieve
    contextos = recuperar_contexto(query, top_k=top_k)

    if verbose:
        print(f"📥 Contextos recuperados ({len(contextos)}):")
        for c in contextos:
            print(f"   [{c['score']:.3f}] {c['titulo']}")
        print()

    # 2. Augment
    prompt = construir_prompt_rag(query, contextos)

    # 3. Generate
    respuesta = invocar_claude(prompt, system=SYSTEM_RAG)
    return respuesta


# Primera prueba del pipeline completo
query = "¿Cuántos días de vacaciones tengo si llevo 6 años en la empresa?"
print(f"❓ Pregunta: {query}")
print("\n🤖 Respuesta RAG:")
print("-" * 60)
print(rag(query, verbose=True))

❓ Pregunta: ¿Cuántos días de vacaciones tengo si llevo 6 años en la empresa?

🤖 Respuesta RAG:
------------------------------------------------------------
📥 Contextos recuperados (3):
   [0.517] Política de Vacaciones
   [0.242] Gastos y Viajes de Empresa
   [0.172] Política de Teletrabajo

Según la **Política de Vacaciones (POL-001)**, como llevas **6 años en la empresa**, tienes derecho a **27 días hábiles de vacaciones anuales**.

Esto se debe a que la política establece que los empleados con más de 5 años en la empresa tienen derecho a 27 días, mientras que los empleados con menos antigüedad tienen 22 días.

**Puntos importantes a recordar:**
- Debes solicitar las vacaciones con al menos 2 semanas de antelación a través del portal de RRHH
- No puedes acumular más de 10 días no disfrutados al siguiente año


## 10. Comparar: con RAG vs sin RAG

In [ ]:
def sin_rag(query: str) -> str:
    """Responde sin contexto — solo con conocimiento del modelo."""
    system = "Eres un asistente de RRHH de TechCorp. Responde con la información que conozcas."
    return invocar_claude(query, system=system)


preguntas = [
    "¿Cuánto me reembolsan por formación al año?",
    "¿Cuántos días puedo teletrabajar si llevo 8 meses en la empresa?",
    "¿Cuándo se realizan las evaluaciones de desempeño?",
]

for pregunta in preguntas:
    print(f"\n{'='*65}")
    print(f"❓ PREGUNTA: {pregunta}")
    print(f"{'='*65}")

    print("\n🚫 SIN RAG (solo conocimiento del modelo):")
    print("-" * 40)
    print(textwrap.fill(sin_rag(pregunta), width=80))

    print("\n✅ CON RAG (contexto de políticas internas):")
    print("-" * 40)
    print(textwrap.fill(rag(pregunta), width=80))


❓ PREGUNTA: ¿Cuánto me reembolsan por formación al año?

🚫 SIN RAG (solo conocimiento del modelo):
----------------------------------------
No tengo acceso a información específica sobre las políticas de reembolso de
formación de TechCorp. Para obtener una respuesta precisa, te recomiendo:  1.
**Consultar el manual de empleado** - Generalmente contiene detalles sobre
beneficios de capacitación 2. **Contactar directamente a RRHH** - El equipo
puede darte los montos exactos y condiciones 3. **Revisar tu contrato** - Puede
incluir cláusulas sobre desarrollo profesional 4. **Preguntar a tu gerente** -
Podría tener información sobre presupuestos de tu departamento  ¿Hay algo más en
lo que pueda ayudarte sobre políticas generales de la empresa?

✅ CON RAG (contexto de políticas internas):
----------------------------------------
Según el **Documento 1 — Beneficios y Compensación (POL-003)**, TechCorp ofrece
**500€ anuales para formación y certificaciones profesionales**.  Este
presupuesto e

## 11. Manejo de preguntas fuera del dominio

In [ ]:
# Preguntas que el sistema RAG debe rechazar o aclarar
preguntas_fuera = [
    "¿Cuál es la capital de Francia?",
    "¿Cuánto cuesta el seguro dental?",  # No está en los documentos
]

for pregunta in preguntas_fuera:
    print(f"\n❓ Pregunta: '{pregunta}'")
    print("Respuesta RAG:")
    print(rag(pregunta, verbose=False))
    print()


❓ Pregunta: '¿Cuál es la capital de Francia?'
Respuesta RAG:
No tengo esa información en los documentos de política interna de TechCorp que me han proporcionado. 

Mi rol es asistir con preguntas sobre las políticas internas de la empresa, como gastos y viajes, seguridad informática, evaluaciones de desempeño, etc.

Si tienes alguna pregunta relacionada con las políticas de TechCorp, estaré encantado de ayudarte. 😊


❓ Pregunta: '¿Cuánto cuesta el seguro dental?'
Respuesta RAG:
No tengo esa información en los documentos de política interna que me han proporcionado. 

Los documentos disponibles cubren políticas sobre gastos y viajes de empresa, seguridad informática y onboarding de nuevos empleados, pero no incluyen detalles sobre beneficios de seguros dentales.

Te recomiendo que contactes directamente con el departamento de RRHH o consultes el manual de beneficios de empleados para obtener información sobre la cobertura dental y sus costos.



## 12. Añadir filtrado por score mínimo

In [ ]:
def rag_con_umbral(query: str, top_k: int = 3, score_minimo: float = 0.5) -> str:
    """
    Pipeline RAG con filtrado por umbral de similitud.
    Si ningún contexto supera el umbral, avisa al usuario.
    """
    contextos = recuperar_contexto(query, top_k=top_k)

    # Filtrar por score mínimo
    contextos_filtrados = [c for c in contextos if c["score"] >= score_minimo]

    print(f"   Contextos totales   : {len(contextos)}")
    print(f"   Contextos filtrados : {len(contextos_filtrados)} (score >= {score_minimo})")

    if not contextos_filtrados:
        return "Lo siento, no encontré información relevante en las políticas de TechCorp para responder tu pregunta."

    prompt = construir_prompt_rag(query, contextos_filtrados)
    return invocar_claude(prompt, system=SYSTEM_RAG)


print("Test con pregunta relevante:")
print(rag_con_umbral("¿Qué pasa con el ticket restaurante?", score_minimo=0.5))

print("\nTest con pregunta fuera de dominio:")
print(rag_con_umbral("¿Cuál es el mejor framework de Python?", score_minimo=0.65))

## 13. Chatbot RAG con historial de conversación

In [ ]:
class ChatbotRAG:
    """Chatbot RAG con memoria de conversación."""

    def __init__(self):
        self.historial = []

    def preguntar(self, query: str, top_k: int = 3) -> str:
        contextos = recuperar_contexto(query, top_k=top_k)
        contexto_str = "\n\n".join(
            f"[{c['titulo']}]: {c['texto']}" for c in contextos
        )

        # Añadir historial al prompt
        historial_str = ""
        if self.historial:
            pares = [f"Usuario: {h['q']}\nAsistente: {h['a']}" for h in self.historial[-3:]]
            historial_str = "\n\n".join(pares) + "\n\n"

        prompt = f"""--- CONTEXTO DE POLÍTICAS ---
{contexto_str}
--- FIN DEL CONTEXTO ---

{historial_str}Usuario: {query}
Asistente:"""

        respuesta = invocar_claude(prompt, system=SYSTEM_RAG)
        self.historial.append({"q": query, "a": respuesta})
        return respuesta

    def reset(self):
        self.historial = []
        print("🔄 Historial reiniciado")


# Simular conversación multi-turno
bot = ChatbotRAG()

conversacion = [
    "¿Cuántos días de teletrabajo puedo hacer?",
    "¿Y si acabo de incorporarme hace 2 meses?",
    "¿Qué herramienta uso para pedir vacaciones?",
]

for turno, pregunta in enumerate(conversacion, 1):
    print(f"\n--- Turno {turno} ---")
    print(f"👤 Usuario: {pregunta}")
    print(f"🤖 Asistente: {bot.preguntar(pregunta)}")

## 14. Resumen y próximos pasos

En este notebook construiste un pipeline RAG completo:

✅ **Ingesta**: carga y chunking de documentos con solapamiento  
✅ **Indexación**: embeddings con Amazon Titan + índice FAISS  
✅ **Retrieval**: búsqueda semántica de los chunks más relevantes  
✅ **Augmentation**: construcción del prompt con contexto recuperado  
✅ **Generation**: respuesta fundamentada con Claude vía Bedrock  
✅ **Evaluación**: comparación con/sin RAG y filtrado por umbral  
✅ **Chatbot**: conversación multi-turno con memoria  

### Extensiones para producción

| Componente | Alternativa gestionada en AWS |
|------------|-------------------------------|
| FAISS local | Amazon OpenSearch con k-NN |
| Documentos locales | Amazon S3 + AWS Glue |
| Pipeline manual | Amazon Bedrock Knowledge Bases |
| Chatbot ad-hoc | Amazon Bedrock Agents |

---

**Siguiente notebook →** Agentes y uso de herramientas con Bedrock Agents